# Q9

In [1]:
import numpy as np

# Set print options for readability
np.set_printoptions(precision=4, suppress=True)

# Define the matrix M
M = np.array([
    [1, 2, 3],
    [4, 5, 6]
])

# Compute the "thin" SVD
# U will be (2, 2), s will be (2,), Vt will be (2, 3)
U, s, Vt = np.linalg.svd(M, full_matrices=False)

# Get the first singular value (sigma_1)
sigma_1 = s[0]

# Get the first left singular vector (u_1)
# U is (2, 2), so u_1 is the first column
u_1 = U[:, 0:1]  # Shape (2, 1)

# Get the first right singular vector (v_1^T)
# Vt is (2, 3), so v_1^T is the first row
v_1_T = Vt[0:1, :]  # Shape (1, 3)

# Calculate the rank-1 approximation
M_1 = sigma_1 * (u_1 @ v_1_T)

# --- Print the results ---
print(f"Matrix M:\n{M}\n")
print("-" * 30)
print(f"sigma_1 (s[0]): {sigma_1:.4f}\n")
print(f"u_1 (U[:, 0]):\n{u_1}\n")
print(f"v_1^T (Vt[0, :]):\n{v_1_T}\n")
print("-" * 30)
print(f"Best Rank-1 Approximation (M_1):\n{M_1}")


Matrix M:
[[1 2 3]
 [4 5 6]]

------------------------------
sigma_1 (s[0]): 9.5080

u_1 (U[:, 0]):
[[-0.3863]
 [-0.9224]]

v_1^T (Vt[0, :]):
[[-0.4287 -0.5663 -0.7039]]

------------------------------
Best Rank-1 Approximation (M_1):
[[1.5745 2.0801 2.5857]
 [3.7594 4.9664 6.1735]]


In [26]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from scipy.spatial.distance import pdist, squareform
from sklearn.preprocessing import MinMaxScaler, StandardScaler

# Set default figure size for visibility
plt.rcParams['figure.figsize'] = (4, 5)

def load_animal_names(filepath='classes.txt'):
    """Loads animal names from classes.txt."""
    with open(filepath, 'r') as f:
        # Read names, strip whitespace
        names = [line.strip() for line in f.readlines()]
        # Split '50.name' into ['50', 'name'] and take 'name'
        # Also replace '+' with ' ' for names like 'grizzly+bear'
        names = [name.split('.')[-1].replace('+', ' ') for name in names]
    return names

def plot_embedding(Z, names, title, filename):
    """Creates and saves a labeled scatter plot for a 2D embedding."""
    print(f"Generating plot: {filename}...")
    plt.figure()
    plt.scatter(Z[:, 0], Z[:, 1])
    
    # Add labels for each point
    for i, name in enumerate(names):
        plt.text(Z[i, 0] + 0.01, Z[i, 1] + 0.01, name, fontsize=8)
        
    plt.title(title, fontsize=16)
    plt.xlabel('Component 1')
    plt.ylabel('Component 2')
    plt.grid(True)
    
    # Save the figure
    plt.tight_layout()
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    plt.close()

def compute_distortion(X_original, Z_embedded):
    """
    Computes the average distortion of an embedding.
    X_original: (n, d) original data
    Z_embedded: (n, 2) embedded data
    """
    
    # Get pairwise distances for original space (N*(N-1)/2 vector)
    D = pdist(X_original, metric='euclidean')
    
    # Get pairwise distances for embedded space (N*(N-1)/2 vector)
    D_hat = pdist(Z_embedded, metric='euclidean')
    
    # Add a small epsilon to avoid division by zero
    # This handles cases where two points are identical in either space
    epsilon = 1e-12
    D[D == 0] = epsilon
    D_hat[D_hat == 0] = epsilon
    
    # Calculate scaling factor c (sum of all pairwise distances)
    c = np.sum(D) / np.sum(D_hat)
    
    # Calculate scaled embedded distances
    scaled_D_hat = c * D_hat
    
    # Calculate distortion for each pair (i, j) with i < j
    Delta_values = np.maximum(D / scaled_D_hat, scaled_D_hat / D)
    
    # Return the mean distortion
    return np.mean(Delta_values)

def main():
    # --- Load Data ---
    print("Loading data...")
    # Load the 50x85 feature matrix
    X = np.loadtxt('predicate-matrix-continuous.txt')
    stand_scaler = StandardScaler()
    #X = stand_scaler.fit_transform(X)
    # Get min and max values
    X_min = np.abs(np.min(X))
    #X = (X+X_min)/(np.max(X)+X_min)
    X_mean = np.mean(X)
    X_std = np.std(X)
    X_median = np.median

    # Handle the case where X_std is 0
    if X_std == 0:
        X = np.zeros_like(X)
    else:
    # Apply the Z-score formula
        X = ((X - X_mean) / X_std)
    print(f"X shape: {X.shape}")
    # Load the 50 animal names
    animal_names = load_animal_names('classes.txt')
    print(f"animal_names: {len(animal_names)}")
    
    # Dictionary to store embeddings and distortion results
    embeddings = {}
    distortions = {}

    # --- Part (a): PCA Visualization ---
    print("\n--- Part (a): PCA ---")
    pca = PCA(n_components=2, random_state=42)
    Z_pca = pca.fit_transform(X)
    embeddings['PCA'] = Z_pca
    
    # Plot and save PCA
    plot_embedding(Z_pca, animal_names, 
                   'PCA Projection of Animals (2D)', 
                   'pca_projection.png')

    # --- Part (b): t-SNE Visualization ---
    print("\n--- Part (b): t-SNE ---")
    perplexities = [5, 10, 25, 49]
    
    for p in perplexities:
        print(f"Running t-SNE with perplexity={p}...")
        tsne = TSNE(n_components=2, perplexity=p, random_state=42, 
                    init='pca', learning_rate='auto')
        Z_tsne = tsne.fit_transform(X)
        
        key = f't-SNE (p={p})'
        embeddings[key] = Z_tsne
        
        # Plot and save t-SNE
        plot_embedding(Z_tsne, animal_names, 
                       f't-SNE Projection (Perplexity={p})', 
                       f'tsne_perplexity_{p}.png')

    # --- Part (c): Distortion Calculation ---
    print("\n--- Part (c): Distortion Analysis ---")
    for key, Z in embeddings.items():
        distortion = compute_distortion(X, Z)
        distortions[key] = distortion
        print(f"Average Distortion for {key}: {distortion:.4f}")
        
    # Find the best embedding (lowest distortion)
    best_embedding = min(distortions, key=distortions.get)
    min_distortion = distortions[best_embedding]
    
    print("\n--- Summary ---")
    print(f"The embedding with the lowest average distortion is: {best_embedding}")
    print(f"Minimum Distortion: {min_distortion:.4f}")

In [27]:
main()

Loading data...
X shape: (50, 85)
animal_names: 50

--- Part (a): PCA ---
Generating plot: pca_projection.png...


/tmp/ipykernel_324524/475806567.py:37: UserWarning: Glyph 9 (	) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_324524/475806567.py:38: UserWarning: Glyph 9 (	) missing from font(s) DejaVu Sans.
  plt.savefig(filename, dpi=300, bbox_inches='tight')



--- Part (b): t-SNE ---
Running t-SNE with perplexity=5...
Generating plot: tsne_perplexity_5.png...


/tmp/ipykernel_324524/475806567.py:37: UserWarning: Glyph 9 (	) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_324524/475806567.py:38: UserWarning: Glyph 9 (	) missing from font(s) DejaVu Sans.
  plt.savefig(filename, dpi=300, bbox_inches='tight')


Running t-SNE with perplexity=10...
Generating plot: tsne_perplexity_10.png...


...

Running t-SNE with perplexity=25...
Generating plot: tsne_perplexity_25.png...


/tmp/ipykernel_324524/475806567.py:37: UserWarning: Glyph 9 (	) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_324524/475806567.py:38: UserWarning: Glyph 9 (	) missing from font(s) DejaVu Sans.
  plt.savefig(filename, dpi=300, bbox_inches='tight')


Running t-SNE with perplexity=49...
Generating plot: tsne_perplexity_49.png...


/tmp/ipykernel_324524/475806567.py:37: UserWarning: Glyph 9 (	) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_324524/475806567.py:38: UserWarning: Glyph 9 (	) missing from font(s) DejaVu Sans.
  plt.savefig(filename, dpi=300, bbox_inches='tight')



--- Part (c): Distortion Analysis ---
Average Distortion for PCA: 1.8313
Average Distortion for t-SNE (p=5): 1.9232
Average Distortion for t-SNE (p=10): 1.6107
Average Distortion for t-SNE (p=25): 1.5793
Average Distortion for t-SNE (p=49): 1.7234

--- Summary ---
The embedding with the lowest average distortion is: t-SNE (p=25)
Minimum Distortion: 1.5793


In [31]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from scipy.spatial.distance import pdist, squareform


plt.rcParams['figure.figsize'] = (6, 8)
def load_animal_names(filepath='classes.txt'):
    """Loads animal names from classes.txt."""
    with open(filepath, 'r') as f:
        # Read names, strip whitespace
        names = [line.strip() for line in f.readlines()]
        # Split '50.name' into ['50', 'name'] and take 'name'
        # Also replace '+' with ' ' for names like 'grizzly+bear'
        names = [name.split('.')[-1].replace('+', ' ') for name in names]
    return names

def compute_distortion(X_original, Z_embedded):
    """
    Computes the average distortion of an embedding.
    X_original: (n, d) original data
    Z_embedded: (n, 2) embedded data
    """
    
    # Get pairwise distances for original space (N*(N-1)/2 vector)
    D = pdist(X_original, metric='euclidean')
    
    # Get pairwise distances for embedded space (N*(N-1)/2 vector)
    D_hat = pdist(Z_embedded, metric='euclidean')
    
    # Add a small epsilon to avoid division by zero
    epsilon = 1e-12
    D[D == 0] = epsilon
    D_hat[D_hat == 0] = epsilon
    
    # Calculate scaling factor c (sum of all pairwise distances)
    c = np.sum(D) / np.sum(D_hat)
    
    # Calculate scaled embedded distances
    scaled_D_hat = c * D_hat
    
    # Calculate distortion for each pair (i, j) with i < j
    Delta_values = np.maximum(D / scaled_D_hat, scaled_D_hat / D)
    
    # Return the mean distortion
    return np.mean(Delta_values)

def main():
    # --- Load Data ---
    print("Loading data...")
    # Load the 50x85 feature matrix
    X = np.loadtxt('predicate-matrix-continuous.txt')
    # Load the 50 animal names
    animal_names = load_animal_names('classes.txt')
    
    # --- Generate Embeddings ---
    # We will store embeddings and their titles in a list
    embeddings_to_plot = []
    
    print("\n--- Part (a): PCA ---")
    pca = PCA(n_components=2, random_state=42)
    Z_pca = pca.fit_transform(X)
    embeddings_to_plot.append(('PCA Projection', Z_pca))

    print("\n--- Part (b): t-SNE ---")
    perplexities = [5, 10, 25, 49]
    for p in perplexities:
        print(f"Running t-SNE with perplexity={p}...")
        tsne = TSNE(n_components=2, perplexity=p, random_state=42, 
                    init='pca', learning_rate='auto')
        Z_tsne = tsne.fit_transform(X)
        embeddings_to_plot.append((f't-SNE (Perplexity={p})', Z_tsne))

    # --- Setup Plotting Styles ---
    num_animals = len(animal_names)
    
    # Create 50 unique colors using a colormap
    colors = plt.cm.turbo(np.linspace(0, 1, num_animals))
    
    # Create a list of markers to cycle through
    # We don't have 50 unique markers, so we cycle
    markers = ['o', 'v', '^', '<', '>', 's', 'p', '*', 'h', 'D', 'd', 'X', 'P']
    
    # --- Create 3x2 Subplot Grid ---
    # (width, height) in inches. (15, 21) is large but needed for 6 plots + legend
    fig, axes = plt.subplots(3, 2, figsize=(8, 10))
    
    # Flatten the 3x2 array for easy 1D iteration
    axes_flat = axes.flat
    
    print("\nGenerating subplots...")
    
    # --- Plot the 5 embeddings ---
    for i, (title, Z) in enumerate(embeddings_to_plot):
        ax = axes_flat[i]
        
        # Plot each animal with its unique color and marker
        for j, name in enumerate(animal_names):
            color = colors[j]
            marker = markers[j % len(markers)]
            
            # CRITICAL: Only add the 'label' argument to the first plot (i==0).
            # This is what we will use to build the legend.
            label = name if i == 0 else None
            
            ax.scatter(Z[j, 0], Z[j, 1], 
                       color=color, 
                       marker=marker, 
                       label=label, 
                       s=50) # s=50 sets the marker size
        
        ax.set_title(title, fontsize=16)
        ax.set_xlabel('Component 1')
        ax.set_ylabel('Component 2')
        ax.grid(True)

    # --- Create the Legend in the 6th Subplot ---
    print("Generating legend...")
    ax_legend = axes_flat[5]
    
    # Get handles and labels from the first plot (axes_flat[0])
    handles, labels = axes_flat[0].get_legend_handles_labels()
    
    # Turn off the axis (box, ticks, etc.) for the legend subplot
    ax_legend.axis('off')
    
    # Create the legend in the center of the 6th subplot
    # We use ncol=5 to make the legend wide instead of tall
    # 50 items / 5 columns = 10 rows
    ax_legend.legend(handles, labels, loc='center', ncol=4, fontsize=9)

    # --- Finalize and Save ---
    # Use tight_layout to prevent titles and labels from overlapping
    plt.tight_layout()
    
    # Save the combined figure
    save_filename = 'embeddings_grid_with_legend.png'
    plt.savefig(save_filename, dpi=150,)# bbox_inches='tight')
    plt.close()
    
    print(f"\nPlot saved as '{save_filename}'")

    # --- Part (c): Distortion Calculation (Unchanged) ---
    print("\n--- Part (c): Distortion Analysis ---")
    distortions = {}
    
    # Store all embeddings in a dict for distortion calculation
    all_embeddings_dict = {title: Z for title, Z in embeddings_to_plot}
        
    for key, Z in all_embeddings_dict.items():
        distortion = compute_distortion(X, Z)
        distortions[key] = distortion
        print(f"Average Distortion for {key}: {distortion:.4f}")
        
    best_embedding = min(distortions, key=distortions.get)
    min_distortion = distortions[best_embedding]
    
    print("\n--- Summary ---")
    print(f"The embedding with the lowest average distortion is: {best_embedding}")
    print(f"Minimum Distortion: {min_distortion:.4f}")


main()

Loading data...

--- Part (a): PCA ---

--- Part (b): t-SNE ---
Running t-SNE with perplexity=5...
Running t-SNE with perplexity=10...
Running t-SNE with perplexity=25...
Running t-SNE with perplexity=49...

Generating subplots...
Generating legend...


/tmp/ipykernel_324524/4068994929.py:135: UserWarning: Glyph 9 (	) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_324524/4068994929.py:139: UserWarning: Glyph 9 (	) missing from font(s) DejaVu Sans.
  plt.savefig(save_filename, dpi=150,)# bbox_inches='tight')



Plot saved as 'embeddings_grid_with_legend.png'

--- Part (c): Distortion Analysis ---
Average Distortion for PCA Projection: 1.8313
Average Distortion for t-SNE (Perplexity=5): 1.8602
Average Distortion for t-SNE (Perplexity=10): 1.7101
Average Distortion for t-SNE (Perplexity=25): 1.5683
Average Distortion for t-SNE (Perplexity=49): 1.7370

--- Summary ---
The embedding with the lowest average distortion is: t-SNE (Perplexity=25)
Minimum Distortion: 1.5683


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from scipy.spatial.distance import pdist, squareform
import re 

def load_animal_names(filepath='classes.txt'):
    with open(filepath, 'r') as f:
        names = [line.strip() for line in f.readlines()]
        names = [name.split('.')[-1].replace('+', ' ') for name in names]
    return(names)

def compute_distortion(X_original, Z_embedded):
    # calc pairwise distances for original space (n*(n-1)/2 vector)
    D = pdist(X_original, metric='euclidean')

    # calc pairwise distances for embedded space (n*(n-1)/2 vector)
    D_hat = pdist(Z_embedded, metric='euclidean')
    
    # add a small epsilon to avoid division by zero
    epsilon = 1e-12
    D[D == 0] = epsilon
    D_hat[D_hat == 0] = epsilon
    
    # calc scaling factor c (sum of all pairwise distances)
    c = np.sum(D) / np.sum(D_hat)
    
    # calc scaled embedded distances
    scaled_D_hat = c * D_hat
    
    # calculate distortion for each pair (i, j) with i < j
    Delta_values = np.maximum(D / scaled_D_hat, scaled_D_hat / D)
    
    return(np.mean(Delta_values))

def main():
    # read predicate-matrix-continuous.txt
    X = np.loadtxt('predicate-matrix-continuous.txt')
    # read classes.txt
    animal_names = load_animal_names('classes.txt')
    
    # initalize list to hold embeddings and their titles
    embeddings_to_plot = []
    
    print("\n--- Part (a): PCA ---")
    pca = PCA(n_components=2, random_state=42)
    Z_pca = pca.fit_transform(X)
    embeddings_to_plot.append(('PCA Projection', Z_pca))

    print("\n--- Part (b): t-SNE ---")
    perplexities = [5, 10, 25, 49]
    for p in perplexities:
        print(f"Running t-SNE with perplexity={p}...")
        tsne = TSNE(n_components=2, perplexity=p, random_state=42, 
                    init='pca', learning_rate='auto')
        Z_tsne = tsne.fit_transform(X)
        embeddings_to_plot.append((f't-SNE (Perplexity={p})', Z_tsne))

    # --- Setup Plotting Styles ---
    num_animals = len(animal_names)
    
    # create list of unique colors using a colormap
    colors = plt.cm.turbo(np.linspace(0, 1, num_animals))
    
    # create a list of markers
    markers = ['o', 'v', '^', '<', '>', 's', 'p', '*', 'h', 'D', 'd', 'X', 'P']
    
    # --- Create One Plot Per Model ---
    print("\nGenerating individual plots with shared legend space...")
    
    for title, Z in embeddings_to_plot:
        # --- Create Figure ---
        # Set figsize to (7.5, 9) which fits well on 8.5x11 paper with margins
        fig = plt.figure(figsize=(7.5, 9))
        
        # --- Create Subplot Axes using subplot2grid ---
        # (rows, cols)
        grid_shape = (3, 3)
        
        # Top plot: starts at (0,0), spans 2 rows, spans 3 columns
        ax_plot = plt.subplot2grid(grid_shape, (0, 0), rowspan=2, colspan=3)
        
        # Bottom legend: starts at (2,0), spans 1 row, spans 3 columns
        ax_legend = plt.subplot2grid(grid_shape, (2, 0), rowspan=1, colspan=3)
        
        # --- Plot the Scatter Data ---
        for j, name in enumerate(animal_names):
            color = colors[j]
            marker = markers[j % len(markers)]
            
            ax_plot.scatter(Z[j, 0], Z[j, 1], 
                            color=color, 
                            marker=marker, 
                            label=name, 
                            s=40) # s=40 is a reasonable marker size
        
        ax_plot.set_title(title, fontsize=16)
        ax_plot.set_xlabel('Component 1')
        ax_plot.set_ylabel('Component 2')
        ax_plot.grid(True)
        
        # --- Create the Legend in the Bottom Subplot ---
        # Get handles and labels from the plot
        handles, labels = ax_plot.get_legend_handles_labels()
        
        # Turn off the axis for the legend subplot
        ax_legend.axis('off')
        
        # Create the legend
        # ncol=5 fits 50 labels in 10 rows
        ax_legend.legend(handles, labels, loc='center', ncol=4, fontsize=9)
        
        # --- Finalize and Save ---
        # Use tight_layout to ensure no overlap
        plt.tight_layout()
        
        # Create a clean filename
        # e.g., "t-SNE (Perplexity=5)" -> "tsne_perplexity_5.png"
        clean_title = title.lower()
        clean_title = re.sub(r'[^\w\s-]', '', clean_title) # Remove non-alphanumeric
        clean_title = re.sub(r'[\s-]+', '_', clean_title).strip('_') # Replace space/dash with _
        filename = f"{clean_title}.png"
        
        plt.savefig(filename, dpi=150) # 150 dpi is good for print
        plt.close(fig) # Close the figure to free memory
        
        print(f"Saved plot: {filename}")

    # --- Part (c): Distortion Calculation (Unchanged) ---
    print("\n--- Part (c): Distortion Analysis ---")
    distortions = {}
    
    # store all embeddings in a dict for distortion calculation
    all_embeddings_dict = {title: Z for title, Z in embeddings_to_plot}
        
    for key, Z in all_embeddings_dict.items():
        distortion = compute_distortion(X, Z)
        distortions[key] = distortion
        print(f"Average Distortion for {key}: {distortion:.4f}")
        
    best_embedding = min(distortions, key=distortions.get)
    min_distortion = distortions[best_embedding]
    
    print(f"lowest average distortion is: {best_embedding}")
    print(f"minimum distortion: {min_distortion:.4f}")


main()

Loading data...

--- Part (a): PCA ---

--- Part (b): t-SNE ---
Running t-SNE with perplexity=5...
Running t-SNE with perplexity=10...
Running t-SNE with perplexity=25...
Running t-SNE with perplexity=49...

Generating individual plots with shared legend space...


/tmp/ipykernel_324524/1918922477.py:130: UserWarning: Glyph 9 (	) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_324524/1918922477.py:139: UserWarning: Glyph 9 (	) missing from font(s) DejaVu Sans.
  plt.savefig(filename, dpi=150) # 150 dpi is good for print


Saved plot: pca_projection.png
Saved plot: t_sne_perplexity5.png
Saved plot: t_sne_perplexity10.png
Saved plot: t_sne_perplexity25.png
Saved plot: t_sne_perplexity49.png

--- Part (c): Distortion Analysis ---
Average Distortion for PCA Projection: 1.8313
Average Distortion for t-SNE (Perplexity=5): 1.8602
Average Distortion for t-SNE (Perplexity=10): 1.7101
Average Distortion for t-SNE (Perplexity=25): 1.5683
Average Distortion for t-SNE (Perplexity=49): 1.7370

--- Summary ---
The embedding with the lowest average distortion is: t-SNE (Perplexity=25)
Minimum Distortion: 1.5683
